# Notebook overview: R8_4-CO_centralities_exploration.ipynb

## What this notebook does
Explores node-level centrality distributions for observed and counterfactual Colorado networks. It computes/loads degree, weighted degree/strength, clustering, closeness, and betweenness-style summaries to compare pre-disaster, post-disaster, and randomized graphs.

## Inputs used
- Observed graph pickles under ../Data/stop_df_perday_CO/graphs_POI_weighted/
- Counterfactual graph pickles and survivor outputs under graphs_POI_weighted/boots_* directories
- Network metric CSVs generated by the random-removal notebooks

## Outputs created
- Node centrality tables
- Network-level summary tables
- Exploratory plots of centrality distributions and pre/post/random comparisons

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install statsmodels 

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import seaborn as sns

In [ ]:
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")

In [ ]:
revision = "R11"

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

In [ ]:
import sys
import os

sys.path.append(os.getcwd())

# Testing Hypothesis:

## 1
H1: Tie preservation is not random with respect to network position.
Nodes with higher structural importance in the pre-disaster network will be disproportionately preserved post-disaster relative to the random survival baseline.

H1b Holding demographics and distance travelled constant, individuals with higher pre-disaster centrality (strength, clustering, closeness) are more likely to remain active in the observed post-disaster network than expected under the random survival baseline.

Degree Strength (exposure proxy)
Hypothesis: Nodes with degree strength will show higher survival probability because they might have access to more resources and higher embededness.

Clustering coefficient (bonding proxy)
Hypothesis: Highly clustered nodes (embedded in tight triads) will show higher survival probability.(Coleman-style closure).

Closeness centrality (integration proxy)
Hypothesis: Nodes with higher closeness are more likely to remain active because they are embedded in well-integrated components.

Are high-centrality people the ones who remain post (vs random)? 

Binary survival model :

POST: $Pr(S_i^{obs}=1)=\text{logit}^{-1}(\alpha + \gamma_1 C_i^{pre} + \gamma_2 H_i^{pre} + \gamma_3 Dist_i^{pre} + \gamma_4 Demog_i)$

RANDOM: $Pr(S_i^{rand}=1)=\text{logit}^{-1}(\alpha + \delta_1 C_i^{pre} + \delta_2 H_i^{pre} + \delta_3 Dist_i^{pre} + \delta_4 Demog_i)$

OR

Create outcome $Y_i = 1$ if observed-post survivor, $Y_i = 0$ if random survivor (within a matched set / weighting so totals comparable):

$Pr(Y_i=1)=\text{logit}^{-1}(\alpha + \theta_1 C_i^{pre} + \theta_2 H_i^{pre} + \theta_3 Similarity_i + \theta_4 Dist_i + \theta_5 Demog_i)$

Interpretation: which features make a survivor more likely observed than null.

H2, Who has higher centrality in pre? 
Individuals with higher third-place category entropy pre-disaster have higher pre-disaster degree centrality (especially strength; potentially closeness), because they participate in more distinct interaction contexts.
Individuals with more concentrated category profiles (lower entropy / stronger preference focus) have higher clustering (closure) because repeated co-presence occurs within fewer stable settings (more triadic closure)

$C_i^{pre} = \alpha + \beta_1 H_i^{pre} + \beta_2 Dist_i^{pre} + \beta_3 Demog_i + \epsilon_i$; $H_i$ - Entropy

$\beta_1 > 0$ for strength; clustering and closeness

## Individual level unique category level interactio- entropy, re-weighted by global baseline probabilities.
Using dyad level data (who visits larger variety of TP categories with someone else)

In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}.csv")
dyad_poi_wide = pd.read_csv(dyad_path)

In [ ]:
dyad_poi_wide.columns

In [ ]:
dyad_poi_wide["pre_sum"]  = pd.to_numeric(dyad_poi_wide["pre_sum"], errors="coerce").fillna(0)
dyad_poi_wide["post_sum"] = pd.to_numeric(dyad_poi_wide["post_sum"], errors="coerce").fillna(0)

In [ ]:
# Keep only rows where there was any interaction pre or post
dyad_nonzero = dyad_poi_wide.loc[(dyad_poi_wide["pre_sum"] > 0) | (dyad_poi_wide["post_sum"] > 0)].copy()

# Symmetrize: create an ego-alter view for BOTH directions
ego_i = dyad_nonzero.rename(columns={"user_i": "user", "user_j": "alter"})
ego_j = dyad_nonzero.rename(columns={"user_j": "user", "user_i": "alter"})
ego = pd.concat([ego_i, ego_j], ignore_index=True)

# Aggregate per user x subcategory
user_subcat = (
    ego
    .groupby(["user", "SUB_CATEGORY"], as_index=False)
    .agg(
        meetings=("alter", "nunique"),
        pre_disaster_visits=("pre_sum", "sum"),
        post_disaster_visits=("post_sum", "sum"),     
    )
)

user_subcat.head()

In [ ]:
# unique POIs per (user, SUB_CATEGORY) in PRE
u_pois_pre_sc = (
    ego[ego["pre_sum"] > 0]
    .groupby(["user", "SUB_CATEGORY"])["poi"].nunique()
    .rename("unique_pois_pre")
    .reset_index()
)

# unique POIs per (user, SUB_CATEGORY) in POST
u_pois_post_sc = (
    ego[ego["post_sum"] > 0]
    .groupby(["user", "SUB_CATEGORY"])["poi"].nunique()
    .rename("unique_pois_post")
    .reset_index()
)

# merge thepre and post uniques
user_unique_pois_sc = (
    u_pois_pre_sc.merge(u_pois_post_sc, on=["user", "SUB_CATEGORY"], how="outer")
    .fillna(0)
)

user_unique_pois_sc["unique_pois_pre"] = user_unique_pois_sc["unique_pois_pre"].astype(int)
user_unique_pois_sc["unique_pois_post"] = user_unique_pois_sc["unique_pois_post"].astype(int)

user_subcat["user"] = user_subcat["user"].astype(str)
user_unique_pois_sc["user"] = user_unique_pois_sc["user"].astype(str)

user_subcat = user_subcat.merge(
    user_unique_pois_sc,
    on=["user", "SUB_CATEGORY"],
    how="left"
)

user_subcat[["unique_pois_pre","unique_pois_post"]] = (
    user_subcat[["unique_pois_pre","unique_pois_post"]].fillna(0).astype(int)
)

user_subcat.head()

In [ ]:
df = user_subcat.copy()

# --- user totals across ALL subcategories ---
df["user_total_pre"] = df.groupby("user")["pre_disaster_visits"].transform("sum")
df["user_total_post"] = df.groupby("user")["post_disaster_visits"].transform("sum")

# --- shares (category visits / user total) ---
df["share_pre"] = np.where(
    df["user_total_pre"] > 0,
    df["pre_disaster_visits"] / df["user_total_pre"],
    np.nan
)

# for post: only non-evacuated users contribute
df["share_post"] = np.where(
    (df["user_total_post"] > 0),
    df["post_disaster_visits"] / df["user_total_post"],
    np.nan
)

df.head()

In [ ]:
global_interaction = (
    dyad_poi_wide
    .groupby("SUB_CATEGORY", as_index=False)
    .agg(
        total_pre_visits=("pre_sum", "sum"),
        total_post_visits=("post_sum", "sum")
    )
)

pre_total = global_interaction["total_pre_visits"].sum()
post_total = global_interaction["total_post_visits"].sum()

global_interaction["prob_pre"] = global_interaction["total_pre_visits"] / pre_total if pre_total > 0 else np.nan
global_interaction["prob_post"] = global_interaction["total_post_visits"] / post_total if post_total > 0 else np.nan

global_interaction[["SUB_CATEGORY", "total_pre_visits", "total_post_visits", "prob_pre", "prob_post"]].head()


In [ ]:
# 1) merge global probs onto user-category rows
df2 = df.merge(
    global_interaction[["SUB_CATEGORY", "prob_pre", "prob_post"]],
    on="SUB_CATEGORY",
    how="left"
)

In [ ]:
# weight individual probability bu global probability:
df2["w_pre"]  = df2["share_pre"]  * df2["prob_pre"]
df2["w_post"] = df2["share_post"] * df2["prob_post"]


In [ ]:
df2["w_pre_norm"] = df2["w_pre"] / df2.groupby("user")["w_pre"].transform("sum")
df2["w_post_norm"] = df2["w_post"] / df2.groupby("user")["w_post"].transform("sum")

In [ ]:
def entropy_from_col(d, col):
    x = d[col].astype(float)
    x = x[(x > 0) & np.isfinite(x)]
    return float(-(x * np.log(x)).sum()) if len(x) else np.nan

# compute user-level entropy
H_pre = df2.groupby("user").apply(lambda g: entropy_from_col(g, "w_pre_norm")).rename("H_weighted_pre").reset_index()
H_post = df2.groupby("user").apply(lambda g: entropy_from_col(g, "w_post_norm")).rename("H_weighted_post").reset_index()

user_weighted_entropy = H_pre.merge(H_post, on="user", how="left")
user_weighted_entropy.head()

# Merge all user level information in centralities df

In [ ]:
csv_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}.csv")
node_centralities = pd.read_csv(csv_path)

In [ ]:
df = node_centralities.copy()
if "user_x" in df.columns:
    df["user"] = df["user_x"].fillna(df.get("node"))
elif "node" in df.columns:
    df["user"] = df["node"]
else:
    raise ValueError("No user identifier found (expected 'user_x' or 'node').")

df["user"] = df["user"].astype(str)
df["run"] = df["run"].astype(str)


In [ ]:
pre_mask = (df["phase"] == "pre")

pre_user = (
    df.loc[pre_mask, ["user"] + centrality_cols]
    .groupby("user", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={c: f"{c}_pre_mean" for c in centrality_cols})
)

In [ ]:
post_mask = (df["phase"] == "post")

post_user = (
    df.loc[post_mask, ["user"] + centrality_cols]
    .groupby("user", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={c: f"{c}_post_mean" for c in centrality_cols})
)

In [ ]:
rand_mask = (df["phase"] == "random_post")

random_user = (
    df.loc[rand_mask, ["user"] + centrality_cols]
    .groupby("user", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={c: f"{c}_random_mean" for c in centrality_cols})
)

rand_n = (
    df.loc[rand_mask, ["user", "run"]]
    .drop_duplicates()
    .groupby("user")
    .size()
    .rename("n_random_runs_present")
    .reset_index()
)

random_user = random_user.merge(rand_n, on="user", how="left")


R = df.loc[rand_mask, "run"].astype(str).nunique()
random_user["n_random_runs_present"] = random_user["n_random_runs_present"].fillna(0).astype(int)
random_user["random_survival_prob"] = random_user["n_random_runs_present"] / R
random_user["random_evacuation_prob"] = 1 - random_user["random_survival_prob"]
random_user.head()

In [ ]:
user_centrality_table = (
    df.loc[pre_mask, ["user"]]
    .drop_duplicates()
    .merge(pre_user, on="user", how="left")
    .merge(post_user, on="user", how="left")
    .merge(random_user, on="user", how="left")
)

In [ ]:
user_centrality_table.head()

In [ ]:
user_weighted_entropy["user"] = user_weighted_entropy["user"].astype(str)

user_centrality_H = user_centrality_table.merge(
    user_weighted_entropy[["user", "H_weighted_pre", "H_weighted_post"]],
    on="user",
    how="left"
)

In [ ]:
user_centrality_H.head()

In [ ]:
start_date = 20211001
end_date = 20211131
freq_home_path = os.path.join(home_dir, f"freq_home_{start_date}_{end_date}")
home_df = pd.read_csv(freq_home_path)
home_df = home_df.rename(columns={"cuebiq_id": "user", "fips_code": "pre_disaster_fips"})
home_df.columns

In [ ]:
week_dir = os.path.join(graph_poi_dir, "user_evacuations")
output_path = os.path.join(week_dir, f"user_evacuations_{revision}.csv")
user_evac_df = pd.read_csv(output_path, low_memory=False)
user_evac_df.head()
# user_evac_df: columns ["user","evacuation"]
user_evac_df["user"] = user_evac_df["user"].astype(str)

# home_df: columns ["user","pre_disaster_fips"]
home_df = home_df.copy()
home_df["user"] = home_df["user"].astype(str)

ue = user_evac_df.merge(home_df[["user", "pre_disaster_fips"]], on="user", how="left")

ue.head()

In [ ]:
user_centrality_H["user"] = user_centrality_H["user"].astype(str)
ue["user"] = ue["user"].astype(str)
user_centrality_H_evac = user_centrality_H.merge(
    ue[["user", "evacuation", "pre_disaster_fips"]],
    on="user",
    how="left"
)
user_centrality_H_evac["evacuation"] = user_centrality_H_evac["evacuation"].fillna(-1).astype(int)
user_centrality_H_evac.head()

In [ ]:
census_path = os.path.join(geo_dir, "colorado_cbg_census_only.csv")
census = pd.read_csv(census_path)
census["black_percent"] = census["black_alone"] / census["total_population"]
census["white_percent"] = census["white_alone"] / census["total_population"]
census.loc[census["median_income"] < 0, "median_income"] = 1
census.loc[census["median_rent"] < 0, "median_rent"] = 1
census["median_income_log"] = np.log1p(census["median_income"])
census["cbg_fips"] = census["cbg_fips"].astype(str)
census.head()

In [ ]:
user_centrality_H_evac["pre_disaster_fips"] = user_centrality_H_evac["pre_disaster_fips"].replace(r'\.0$', '', regex=True).astype(str)


In [ ]:
user_centrality_H_evac_census = user_centrality_H_evac.merge(census,
    left_on="pre_disaster_fips", right_on="cbg_fips",
    how="left")

In [ ]:
user_centrality_H_evac_census.head()

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
user_centrality_H_evac_census.to_csv(out_path, index=False)
print("Saved:", out_path)

# Regressions

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
user_centrality_H_evac_census = pd.read_csv(out_path)
user_centrality_H_evac_census.head()

In [ ]:
user_centrality_H_evac_census.columns

In [ ]:
sdm_cols = ["total_population","median_age","white_percent","median_income_log","median_rent"]
std_cols_pre = ["H_weighted_pre"] + sdm_cols
#std_cols_post = ["H_weighted_post"] + sdm_cols
# Standardize on observed data
scaler = StandardScaler()
user_centrality_H_evac_census[std_cols_pre] = scaler.fit_transform(user_centrality_H_evac_census[std_cols_pre])
#user_centrality_H_evac_census[std_cols_post] = scaler.fit_transform(user_centrality_H_evac_census[std_cols_post])

In [ ]:
model_specs = [
    ("Post_strength",
     "evacuation ~ strength_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Random_strength",
     "random_evacuation_prob ~ strength_pre_mean + " + " + ".join(std_cols_pre)),

    ("Post_close",
     "evacuation ~ closeness_centrality_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Random_close",
     "random_evacuation_prob ~ closeness_centrality_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Post_cluster",
     "evacuation ~ clustering_coefficient_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Random_cluster",
     "random_evacuation_prob ~ clustering_coefficient_pre_mean + " + " + ".join(std_cols_pre)),

]

all_results = []
fitted_models = {}

for model_name, formula in model_specs:
    print(f"Running model: {model_name}")

    model = smf.ols(formula=formula, data=user_centrality_H_evac_census).fit()

    fitted_models[model_name] = model  

    res_df = extract_model_results(model, model_name)
    all_results.append(res_df)
results_df = pd.concat(all_results, ignore_index=True)
results_df